# 方案 1B (GatewayLlmLogs) — 端到端驗證 notebook

驗證 `azuremonitor` diagnostic 的 `largeLanguageModel.logs=enabled` 設定能否：

- TC-1B-1：對小 non-streaming 請求 → 寫入 `ApiManagementGatewayLlmLog`（或 `AzureDiagnostics where Category=="GatewayLlmLogs"`），SequenceNumber=0..2 三筆 row 完整
- TC-1B-2：對大 non-streaming 請求 (>32 KB raw response) → response 被 chunk 切成多筆 (seq≥2)，重組後等於完整 message
- TC-1B-3：對 streaming SSE 請求 (Kimi-K2.5 reasoning) → `IsStreamCompletion=true`，且 response 內容被 logger 自動重組為完整 assistant message（**這是相對方案 1A 最大的優勢**）
- TC-1B-4：對照組 — 查不存在的 marker → 應拿到「not found」（保證沒有偽造資料）

每個 TC 後面都跟一段 KQL 驗證 cell。

前置：APIM API 已套上 `azuremonitor` API-level diagnostic，且 service-level diagnostic settings 已把 `GatewayLlmLogs` 路由到 LAW。設定請參考 `SOLUTIONS.md` § 方案 1B 啟用步驟。


## Setup


In [ ]:
import os, sys, time, json, textwrap
from getpass import getpass
from openai import OpenAI

# 1. APIM endpoint（與 Solution C notebook 相同）
# 2. 此 API 用 header 名稱 `api-key`（不是 Ocp-Apim-Subscription-Key）
APIM_BASE_URL     = 'https://testaigw01.azure-api.net/kunlenewfoundry01/openai/v1/'
APIM_SUBSCRIPTION = os.environ.get('APIM_SUBSCRIPTION_KEY') or getpass('APIM Subscription Key: ')
MODEL             = 'Kimi-K2.5'

# 3. Log Analytics workspace 設定
# 從 Azure CLI 拿 customerId（GUID）
import subprocess
LAW_NAME = os.environ.get('LAW_NAME', 'log-aigw-5e3hqtbguevmo')
LAW_RG   = os.environ.get('LAW_RG',   'newfoundry01')
LAW_ID   = subprocess.check_output(f'az monitor log-analytics workspace show -n {LAW_NAME} -g {LAW_RG} --query customerId -o tsv', text=True, shell=True).strip()
print('LAW customerId =', LAW_ID)

# 4. 標記 — 注入到 prompt 文字，事後可以 KQL 撈
RUN_ID = f'sol1b-{int(time.time())}'
print('RUN_ID =', RUN_ID)

client = OpenAI(
    base_url        = APIM_BASE_URL,
    api_key         = 'unused',  # 真正的 key 放 default_headers
    default_headers = { 'api-key': APIM_SUBSCRIPTION },
)
print('Setup OK.')


## Helper — `verify_marker_1b`

輪詢 LAW 直到找到含 marker 的 GatewayLlmLogs rows。聚合同 CorrelationId 的所有 row，依 SequenceNumber 排序，重組 request + response，做完整性檢查並印出。

查詢使用 **`AzureDiagnostics where Category=="GatewayLlmLogs"`**（legacy mode；本 lab 的 diagnostic setting 目前用此模式）。如果你的 diagnostic 已經切成 Resource specific (Dedicated) destination type，把 `TABLE_NAME` 改成 `'ApiManagementGatewayLlmLog'`、欄位名換掉 `_s` / `_d` / `_b` 後綴即可。


In [ ]:
TABLE_NAME = 'AzureDiagnostics'  # 改成 'ApiManagementGatewayLlmLog' 即用 Dedicated mode

def _kql_legacy(marker, lookback_min=30):
    return textwrap.dedent(f'''
        AzureDiagnostics
        | where Category == "GatewayLlmLogs"
          and TimeGenerated > ago({lookback_min}m)
          and (requestMessages_s contains "{marker}" or CorrelationId in
               ((AzureDiagnostics | where Category=="GatewayLlmLogs" and TimeGenerated > ago({lookback_min}m) and requestMessages_s contains "{marker}" | distinct CorrelationId)))
        | project TimeGenerated, CorrelationId,
                  seq         = toint(sequenceNumber_d),
                  isStream    = isStreamCompletion_b,
                  modelName   = modelName_s,
                  promptTok   = toint(promptTokens_d),
                  completionT = toint(completionTokens_d),
                  totalTok    = toint(totalTokens_d),
                  reqMsgs     = requestMessages_s,
                  respMsgs    = responseMessages_s
        | order by CorrelationId asc, seq asc
    ''').strip()

def _run_kql(query):
    r = subprocess.run(['az','monitor','log-analytics','query','-w',LAW_ID,'--analytics-query',query,'-o','json'],
                       capture_output=True, text=True, shell=True)
    if r.returncode != 0:
        raise RuntimeError(f'az query failed (rc={r.returncode}): {r.stderr or r.stdout}')
    return json.loads(r.stdout)

def verify_marker_1b(marker, expect_response=True, expect_stream=None, max_wait=240, poll_every=15, lookback_min=30):
    print(f'[verify] marker = {marker}')
    print(f'[verify] LAW   = {LAW_NAME} (customerId={LAW_ID})')
    print(f'[verify] expect_response={expect_response}, expect_stream={expect_stream}, max_wait={max_wait}s')
    waited = 0
    rows = []
    while waited < max_wait:
        rows = _run_kql(_kql_legacy(marker, lookback_min=lookback_min))
        if rows:
            break
        print(f'  ...no rows yet, sleep {poll_every}s (waited {waited}/{max_wait})')
        time.sleep(poll_every); waited += poll_every
    if not rows:
        print('❌ MARKER NOT FOUND in', max_wait, 's')
        return None
    # group by correlationId
    by_corr = {}
    for r in rows:
        by_corr.setdefault(r['CorrelationId'], []).append(r)
    print(f'✅ found {len(rows)} row(s) across {len(by_corr)} correlationId(s)')
    for corr, items in by_corr.items():
        items.sort(key=lambda x: x['seq'] if x['seq'] is not None else -1)
        meta_row = next((x for x in items if x['seq']==0), None)
        req_rows = [x for x in items if x['reqMsgs']]
        resp_rows= [x for x in items if x['respMsgs']]
        seqs     = [x['seq'] for x in items]
        req_full = ''.join(x['reqMsgs'] or '' for x in req_rows)
        resp_full= ''.join(x['respMsgs']or '' for x in resp_rows)
        print('')
        print(f'=== CorrelationId {corr} ===')
        print(f'    rows         : {len(items)} (seq = {seqs})')
        if meta_row:
            print(f'    metadata seq=0 : model={meta_row["modelName"]}, isStream={meta_row["isStream"]}, prompt={meta_row["promptTok"]}, completion={meta_row["completionT"]}, total={meta_row["totalTok"]}')
        else:
            print('    ⚠️  no seq=0 metadata row')
        # checks
        seq_set = set(s for s in seqs if s is not None)
        contiguous = seq_set == set(range(0, max(seq_set)+1)) if seq_set else False
        print(f'    SequenceNumber contiguous from 0 : {"✅" if contiguous else "❌"} (got {sorted(seq_set)})')
        print(f'    request rows   : {len(req_rows)} → reassembled length {len(req_full)} chars')
        if req_full:
            print(f'        head : {req_full[:120]!r}')
        print(f'    response rows  : {len(resp_rows)} → reassembled length {len(resp_full)} chars')
        if resp_full:
            print(f'        head : {resp_full[:120]!r}')
            print(f'        tail : {resp_full[-120:]!r}')
        if expect_response:
            print(f'    expect response: {"✅" if len(resp_full) > 0 else "❌ EMPTY"}')
        if expect_stream is not None and meta_row is not None:
            actual = bool(meta_row['isStream'])
            print(f'    expect_stream={expect_stream} : {"✅" if actual==expect_stream else f"❌ got {actual}"}')
    return by_corr

print('verify_marker_1b ready')


## TC-1B-1 — 短 non-streaming（驗證基本 row 組裝：seq=0,1,2）


In [ ]:
marker = f'{RUN_ID}-tc1'
print('marker =', marker)
resp = client.chat.completions.create(
    model    = MODEL,
    messages = [{'role':'user','content': f'[{marker}] 用一句話說明 Azure API Management。'}],
    max_tokens = 1500,
    stream     = False,
)
print('finish_reason =', resp.choices[0].finish_reason)
_content = resp.choices[0].message.content or ''
print('content len   =', len(_content))
print('content head  =', _content[:200])
print('usage         =', resp.usage)


### KQL 驗證 — TC-1B-1
預期：1 個 CorrelationId、3 筆 row（seq=0 metadata、seq=1 request、seq=2 response）、isStream=False、completion tokens 與 response 內容都不為空。


In [ ]:
verify_marker_1b(marker, expect_response=True, expect_stream=False, max_wait=240)


## TC-1B-2 — 大 non-streaming（驗證 SequenceNumber chunking）


In [ ]:
marker = f'{RUN_ID}-tc2'
print('marker =', marker)
prompt = f'[{marker}] 請輸出一段約 5000 字的繁體中文短篇科幻故事，主題是太空船維修工程師與 AI 助手共同處理引擎異常。請務必盡量寫長，不要少於 3500 字。'
resp = client.chat.completions.create(
    model      = MODEL,
    messages   = [{'role':'user','content': prompt}],
    max_tokens = 3500,
    stream     = False,
)
raw_response_text = resp.choices[0].message.content or ''
print('finish_reason =', resp.choices[0].finish_reason)
print('response chars =', len(raw_response_text))
print('usage          =', resp.usage)


### KQL 驗證 — TC-1B-2
預期：1 個 CorrelationId、≥4 筆 row（seq=0,1,2,3,...）、isStream=False、response **被切成 ≥2 個 chunk**（seq=2,3,...）、重組後 reassembled length 接近 raw response 經 JSON-escape 後的長度（中文 `\uXXXX` 6 倍膨脹）。


In [ ]:
verify_marker_1b(marker, expect_response=True, expect_stream=False, max_wait=300)


## TC-1B-3 — Streaming SSE（⭐ 驗證 1B 對 1A 的關鍵優勢）

Kimi-K2.5 streaming，request 為長推理。**1A 在這個場景只能拿到首封包 (~200 bytes)；1B 應該能拿到完整 assistant content。**


In [ ]:
marker = f'{RUN_ID}-tc3'
print('marker =', marker)
prompt = f'[{marker}] 用繁體中文，仔細推理並逐步說明：如何設計一個高可用的多區域 Azure SQL 失敗轉移架構？包含 read replica、SQL agent job、failover group、application gateway 健康檢查、以及應用層的重試策略。請寫得越詳細越好，至少 3000 字。'
stream = client.chat.completions.create(
    model      = MODEL,
    messages   = [{'role':'user','content': prompt}],
    max_tokens = 3000,
    stream     = True,
)
collected = []
for chunk in stream:
    if chunk.choices and chunk.choices[0].delta and chunk.choices[0].delta.content:
        collected.append(chunk.choices[0].delta.content)
client_side_full = ''.join(collected)
print('client side reassembled chars =', len(client_side_full))
print('first 200 chars =', client_side_full[:200])


### KQL 驗證 — TC-1B-3
預期：1 個 CorrelationId、isStream=**True**、response 內容由 APIM 在 logger 端把 SSE delta 自動重組為**乾淨的 assistant message**（不含 `data: ` SSE wrapper），長度應與 client_side_full 同數量級（3000 token Chinese ≈ 3000–4500 chars）。


In [ ]:
verify_marker_1b(marker, expect_response=True, expect_stream=True, max_wait=300)


## TC-1B-4 — 對照組（marker 不存在 → not found）


In [ ]:
fake_marker = f'{RUN_ID}-tc4-NEVER-SENT'
print('marker =', fake_marker)
result = verify_marker_1b(fake_marker, expect_response=False, max_wait=60, poll_every=20)
print('')
print('expected result == None:', result is None)


## 驗證總結

如果 TC-1B-1 ~ TC-1B-3 都打勾（contiguous seq、reassembled length > 0、isStream 一致），就證明：

1. ✅ APIM `azuremonitor` diagnostic + `largeLanguageModel.logs=enabled` 確實把 LLM 對話路由到 LAW
2. ✅ Chunking + SequenceNumber 機制會自動把 > 32 KB 的 response 切割並可被 KQL 重組
3. ✅ Streaming SSE 會被 APIM logger 自動重組為完整 assistant message（**1B 相對 1A 最大的差異**）
4. ✅ TC-1B-4 證明驗證腳本不會誤報

→ 客戶可直接採用方案 1B，零 policy code、原生 Azure 整合。

若需要對話 > 2 MB / 不能進 LAW / 永久歸檔，再考慮方案 2。
